# T1 brain extraction

Apply a pretrained brain extraction model and threshold its brain probability map.

**Resources:** downloads an example T1, templates and pretrained weights on first use.
These 3-D workflows can take several minutes or longer and substantial memory on CPU.
Assets are cached under `~/.antstorch`. Inspect outputs on your own data.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import os
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = os.environ.get("ANTSTORCH_EXAMPLE_DEVICE", "cpu")
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

## Load the sample MRI or choose a local image

In [ ]:
from pathlib import Path

# Set an absolute path to use your own scalar 3-D T1 MRI instead.
t1_path = None
if t1_path is None:
    t1_path = antstorch.get_antstorch_data("mprage_hippmapp3r")
t1 = ants.image_read(str(t1_path)).clone("float")
assert t1.dimension == 3 and not t1.has_components
assert np.isfinite(t1.numpy()).all()
print("Input:", t1_path, "shape:", t1.shape, "spacing:", t1.spacing)

In [ ]:
with time_step('Brain extraction (probability)'):
    probability = antstorch.brain_extraction(t1, modality="t1", device=device, verbose=True)
assert probability.shape == t1.shape
assert np.isfinite(probability.numpy()).all()
assert probability.numpy().min() >= -1e-6
assert probability.numpy().max() <= 1 + 1e-6
np.testing.assert_allclose(probability.spacing, t1.spacing)
np.testing.assert_allclose(probability.origin, t1.origin)
np.testing.assert_allclose(probability.direction, t1.direction)
mask = ants.threshold_image(probability, 0.5, 1.0, 1, 0)
assert (mask.numpy() > 0).any()
brain = t1 * mask

## Inspect and save
A 0.5 threshold is used for this demonstration. The overlay displays the binary mask. Check it for excluded
brain tissue and retained nonbrain tissue before using the mask downstream.

In [ ]:
ants.plot(t1, overlay=mask, axis=2, nslices=6,
          overlay_cmap="Reds", overlay_alpha=0.5)
output_dir = Path("example_outputs/brain")
output_dir.mkdir(parents=True, exist_ok=True)
for name, image in {"brain_probability": probability, "brain_mask": mask, "brain": brain}.items():
    ants.image_write(image, str(output_dir / f"{name}.nii.gz"))
print("Outputs (matching files are overwritten):", output_dir.resolve())

## Other modalities and T1 variants
Continue with [brain_extraction_modalities.ipynb](brain_extraction_modalities.ipynb)
for T2, T2*, FLAIR, BOLD, FA, PET, MRA and the T1 three-tissue,
hemisphere and lobe variants. Their input requirements and return types differ.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")